# 03e — Scenario A QUBO Construction + Classical Baseline

## Purpose
This notebook takes **Scenario A** (a curated set of clinical trials) and turns it into a **concrete QUBO optimization problem**, then solves it using a **classical baseline**. This gives us:
- a reproducible QUBO artifact (matrix + variable mapping + weights)
- a set of selected trials (bitstring to trial IDs)
- baseline performance metrics we can later compare against QAOA (local + SV1)

## Inputs
- `data/interim/trials_scenarios.parquet`  
  Scenario-ready feature table (benefit / cost / safety / feasibility columns).
- `data/scenarios/scenario_A_trial_ids.csv`  
  The trial universe for Scenario A.

## Outputs (small, GitHub-safe artifacts)
- `data/qubo/scenario_A_qubo.json`
- `data/results/scenario_A_classical_selected_trials.csv`
- `data/results/scenario_A_classical_summary.csv`
- `data/results/scenario_A_best_bitstring.txt`

## Modeling choices (initial)
- We reduce Scenario A down to **N variables** to keep QUBO tractable.
- Objective (minimize): **−benefit + λ_cost·cost + λ_safety·risk**
- Constraint: **select exactly K** trials (penalty form: `P·(Σx − K)²`)


In [2]:
# ============================================================
# Cell 1 — Imports + Configuration
# ============================================================
#
# What this cell does:
#   1) Imports standard libraries (paths, json, numeric + tabular).
#   2) Defines input/output paths used throughout the notebook.
#   3) Defines key "knobs" (N variables, K selection size, weights).
#   4) Provides a simple log() function for consistent timestamps.
#
# Why it matters:
#   - Centralizes knobs and paths to make the notebook reproducible.
#   - Ensures output directories exist before writing artifacts.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
from datetime import datetime

# ----------------------------
# Input paths (from Phase 2)
# ----------------------------
SCENARIOS_PARQUET = Path("data/interim/trials_scenarios.parquet")
SCENARIO_A_IDS    = Path("data/scenarios/scenario_A_trial_ids.csv")

# ----------------------------
# Output paths (GitHub-safe)
# ----------------------------
QUBO_DIR    = Path("data/qubo")
RESULTS_DIR = Path("data/results")

# Create output folders if missing
QUBO_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Optimization "knobs"
# ----------------------------
# Reduce to a manageable number of binary variables for QUBO.
# Keep this small while validating the pipeline; increase later.
N_VARIABLES = 18

# Portfolio size: select exactly K trials (constraint enforced by penalty).
PORTFOLIO_K = 6

# Objective weights (tune later)
LAMBDA_COST   = 1.0     # higher → cost matters more
LAMBDA_SAFETY = 1.0     # higher → safety risk matters more

# Penalty strength for enforcing exactly-K constraint.
# If too small, constraint can be violated. If too large, it dominates objective.
PENALTY_K = 50.0

def log(msg: str) -> None:
    """Timestamped print for progress tracing."""
    print(f"{datetime.now().strftime('%H:%M:%S')} | {msg}")

log("[Cell 2] Imports and config loaded.")

15:56:39 | [Cell 2] Imports and config loaded.


### What Cell 1 Just Did

This cell set up the notebook’s foundation: imports, input/output paths, and the main modeling parameters.  
Most importantly, it defines:
- the reduced QUBO size (`N_VARIABLES`),
- the target portfolio size (`PORTFOLIO_K`),
- cost and safety weights (`LAMBDA_COST`, `LAMBDA_SAFETY`), and
- the penalty strength used to enforce “select exactly K trials” (`PENALTY_K`).

These values are intentionally conservative for a first reproducible baseline.


In [3]:
# ============================================================
# Cell 2 — Load scenario-ready trials table + Scenario A IDs
# ============================================================
#
# What this cell does:
#   1) Loads the scenario-ready table from Parquet (Phase 2 artifact).
#   2) Loads Scenario A trial IDs (CSV).
#   3) Filters the master table down to the Scenario A universe.
#
# Why it matters:
#   - Ensures we only optimize over a scenario-defined subset.
#   - Keeps the pipeline clean: Phase 2 prepares features, Phase 3 uses them.
# ============================================================

import pyarrow.parquet as pq

# --- Validate inputs exist early ---
if not SCENARIOS_PARQUET.exists():
    raise FileNotFoundError(f"Missing required input: {SCENARIOS_PARQUET}")

if not SCENARIO_A_IDS.exists():
    raise FileNotFoundError(f"Missing required input: {SCENARIO_A_IDS}")

# --- Load trials_scenarios (feature-ready table) ---
trials_scenarios = pq.read_table(SCENARIOS_PARQUET).to_pandas()
log(f"[Cell 3] Loaded trials_scenarios with shape: {trials_scenarios.shape}")

# --- Load Scenario A trial IDs ---
df_ids = pd.read_csv(SCENARIO_A_IDS)
if "nct_id" not in df_ids.columns:
    raise KeyError("scenario_A_trial_ids.csv must contain a column named 'nct_id'")

scenario_a_ids = set(df_ids["nct_id"].astype(str))
log(f"[Cell 3] Loaded Scenario A IDs: {len(scenario_a_ids):,}")

# --- Filter master table to Scenario A universe ---
df_a = trials_scenarios[trials_scenarios["nct_id"].astype(str).isin(scenario_a_ids)].copy()
log(f"[Cell 3] Filtered Scenario A subset shape: {df_a.shape}")

df_a.head()


15:56:46 | [Cell 3] Loaded trials_scenarios with shape: (557292, 13)
15:56:46 | [Cell 3] Loaded Scenario A IDs: 34,794
15:56:46 | [Cell 3] Filtered Scenario A subset shape: (34794, 13)


,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score,benefit_score
20,NCT00000124,Collaborative Ocular Melanoma Study (COMS),Unknown status,Phase 3,"[Choroid Neoplasms, Uveitis]","[Brachytherapy, Eye Removal]",[],National Eye Institute (NEI),National Eye Institute (NEI),Global / Multi-Region,4.0,1.0,0.16
312,NCT00000433,Blocking Tumor Necrosis Factor in Ankylosing S...,Completed,Phase 2,"[Spondylitis, Ankylosing]",[Anti-Tumor Necrosis Factor],[United States],National Institute of Arthritis and Musculoske...,National Institute of Arthritis and Musculoske...,Global / Multi-Region,2.0,1.0,0.36
357,NCT00000479,Women's Health Study (WHS): A Randomized Trial...,Completed,Phase 3,"[Cardiovascular Diseases, Cerebrovascular Diso...","[Aspirin, Vitamin E, Placebo]",[],Brigham and Women's Hospital,Brigham and Women's Hospital,Global / Multi-Region,4.0,1.0,0.48
467,NCT00000589,Trial to Reduce Alloimmunization to Platelets ...,Completed,Phase 3,"[Blood Platelets, Hematologic Diseases, Immuni...",[platelet transfusion],[],"National Heart, Lung, and Blood Institute (NHLBI)","National Heart, Lung, and Blood Institute (NHLBI)",Global / Multi-Region,4.0,1.0,0.48
469,NCT00000591,T-Cell Depletion in Unrelated Donor Marrow Tra...,Completed,Phase 3,"[Bone Marrow Transplantation, Graft vs Host Di...",[lymphocyte depletion],[],"National Heart, Lung, and Blood Institute (NHLBI)","National Heart, Lung, and Blood Institute (NHLBI)",Global / Multi-Region,4.0,1.0,0.48


### What Cell 2 Just Did

This cell loaded the Phase 2 scenario-ready dataset and the Scenario A trial ID list, then filtered the master table down to the Scenario A universe.  
From this point forward, all feature normalization, candidate reduction, QUBO construction, and baseline solving happens strictly within Scenario A.


In [4]:
# ============================================================
# Cell 3 — Normalize objective components (benefit, cost, safety)
# ============================================================
#
# What this cell does:
#   1) Detects which columns exist for benefit, cost, and safety.
#   2) Normalizes each component to a comparable 0..1 scale.
#   3) Builds a combined score used to reduce Scenario A to N variables.
#
# Why it matters:
#   - QUBO construction requires numeric objective terms per variable.
#   - Normalization prevents any single raw scale from dominating.
#   - A reduction score allows a controlled downselection for tractability.
# ============================================================

# Candidate column names (robust to earlier notebook variations)
benefit_col_candidates = ["benefit_score", "benefit_norm", "candidate_score"]
cost_col_candidates    = ["estimated_trial_cost", "cost_norm"]
safety_col_candidates  = ["safety_score", "sponsor_safety_score", "safety_norm"]

def pick_first_existing(cols, df):
    """Return the first column name in cols that exists in df; else None."""
    for c in cols:
        if c in df.columns:
            return c
    return None

benefit_col = pick_first_existing(benefit_col_candidates, df_a)
cost_col    = pick_first_existing(cost_col_candidates, df_a)
safety_col  = pick_first_existing(safety_col_candidates, df_a)

log(f"[Cell 4] Using columns: benefit={benefit_col} | cost={cost_col} | safety={safety_col}")

def minmax(s: pd.Series) -> pd.Series:
    """
    Min-max normalize to [0, 1]. Missing values become 0.0.
    If series is constant, return all zeros to avoid divide-by-zero.
    """
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    mn, mx = float(s.min()), float(s.max())
    return (s - mn) / (mx - mn) if mx > mn else pd.Series(0.0, index=s.index)

# ----------------------------
# Benefit normalization
# ----------------------------
# If no explicit benefit column exists, fall back to feasibility as a proxy.
if benefit_col is None:
    if "enrollment_feasibility_score" not in df_a.columns:
        raise KeyError(
            "No benefit column found (benefit_score/candidate_score) "
            "and enrollment_feasibility_score is missing."
        )
    df_a["benefit_work"] = minmax(df_a["enrollment_feasibility_score"])
    benefit_col = "benefit_work"
else:
    df_a["benefit_work"] = minmax(df_a[benefit_col])

# ----------------------------
# Cost normalization
# ----------------------------
# If no cost exists, treat as zero cost (neutral).
if cost_col is None:
    df_a["cost_work"] = 0.0
else:
    # Cost tends to be heavy-tailed; log1p reduces extreme scale effects.
    raw_cost = pd.to_numeric(df_a[cost_col], errors="coerce")
    raw_cost = raw_cost.fillna(raw_cost.median())
    df_a["cost_work"] = minmax(np.log1p(raw_cost).astype(float))

# ----------------------------
# Safety normalization
# ----------------------------
# If no safety exists yet, treat as zero risk (neutral).
if safety_col is None:
    df_a["safety_work"] = 0.0
else:
    df_a["safety_work"] = minmax(df_a[safety_col])

# ----------------------------
# Reduction score (higher = better)
# ----------------------------
# We want high benefit, low cost, low risk.
df_a["score_for_reduction"] = (
    df_a["benefit_work"]
    - LAMBDA_COST   * df_a["cost_work"]
    - LAMBDA_SAFETY * df_a["safety_work"]
)

df_a[["nct_id", "phase", "score_for_reduction", "benefit_work", "cost_work", "safety_work"]].head()


15:56:51 | [Cell 4] Using columns: benefit=benefit_score | cost=estimated_trial_cost | safety=None


,nct_id,phase,score_for_reduction,benefit_work,cost_work,safety_work
20,NCT00000124,Phase 3,-0.941176,0.058824,1.0,0.0
312,NCT00000433,Phase 2,0.352941,0.352941,0.0,0.0
357,NCT00000479,Phase 3,-0.470588,0.529412,1.0,0.0
467,NCT00000589,Phase 3,-0.470588,0.529412,1.0,0.0
469,NCT00000591,Phase 3,-0.470588,0.529412,1.0,0.0


### What Cell 3 Just Did

This cell standardized the objective components needed for optimization. It detects which columns are available for benefit, cost, and safety risk, then normalizes each to a 0–1 scale for comparability.

It then creates a single `score_for_reduction` that rewards higher benefit and penalizes cost and safety risk. This score is used solely to pick a tractable subset of trials for QUBO construction, not to “solve” the optimization problem by itself.


In [5]:
# ============================================================
# Cell 4 — Reduce candidate set + Construct QUBO matrix
# ============================================================
#
# What this cell does:
#   1) Selects the top N_VARIABLES trials by score_for_reduction.
#   2) Builds a QUBO objective:
#         minimize  Σ d_i x_i
#       where d_i = -(benefit) + λ_cost*(cost) + λ_safety*(risk)
#   3) Adds an exactly-K constraint using a quadratic penalty:
#         PENALTY_K * (Σ x_i - K)^2
#   4) Writes a reproducible QUBO artifact to JSON:
#         - Q matrix (dense)
#         - nct_id mapping (variable index -> trial id)
#         - weights + settings
#
# Why it matters:
#   - This creates the canonical QUBO artifact against which both
#     classical and quantum solvers can be compared.
# ============================================================

# --- Reduce Scenario A to a tractable candidate set ---
df_red = (
    df_a.sort_values("score_for_reduction", ascending=False)
        .head(N_VARIABLES)
        .reset_index(drop=True)
)

# Map variable index -> nct_id
nct_ids = df_red["nct_id"].astype(str).tolist()
N = len(nct_ids)
log(f"[Cell 5] Reduced candidate set to N={N} variables")

# --- Build linear objective coefficients (minimization form) ---
# d_i = -(benefit) + λ_cost*cost + λ_safety*safety
d = (
    -df_red["benefit_work"].to_numpy(dtype=float)
    + LAMBDA_COST   * df_red["cost_work"].to_numpy(dtype=float)
    + LAMBDA_SAFETY * df_red["safety_work"].to_numpy(dtype=float)
)

# Initialize dense QUBO matrix
Q = np.zeros((N, N), dtype=float)

# Put linear terms on diagonal
Q[np.diag_indices(N)] = d

# --- Add exactly-K penalty: P*(sum(x)-K)^2 ---
#
# Expand: P*(sum x - K)^2 = P*( (sum x)^2 - 2K(sum x) + K^2 )
#
# (sum x)^2 = sum x_i^2 + 2*sum_{i<j} x_i x_j
# But for binary variables, x_i^2 = x_i.
#
# So:
#   diagonal add: P*(1 - 2K)
#   off-diagonal add: 2P for every pair (i<j)
#
Q[np.diag_indices(N)] += PENALTY_K * (1.0 - 2.0 * PORTFOLIO_K)

for i in range(N):
    for j in range(i + 1, N):
        Q[i, j] += 2.0 * PENALTY_K
        Q[j, i] += 2.0 * PENALTY_K

# --- Persist QUBO artifact to JSON (reproducible) ---
qubo_payload = {
    "scenario": "A",
    "created_utc": datetime.utcnow().isoformat() + "Z",
    "n_variables": int(N),
    "portfolio_k": int(PORTFOLIO_K),
    "lambda_cost": float(LAMBDA_COST),
    "lambda_safety": float(LAMBDA_SAFETY),
    "penalty_k": float(PENALTY_K),
    "nct_ids": nct_ids,
    "Q_dense": Q.tolist(),
}

qubo_path = QUBO_DIR / "scenario_A_qubo.json"
with open(qubo_path, "w") as f:
    json.dump(qubo_payload, f)

log(f"[Cell 5] Wrote QUBO artifact → {qubo_path}")
df_red[["nct_id", "phase", "score_for_reduction"]].head()


15:56:57 | [Cell 5] Reduced candidate set to N=18 variables
15:56:57 | [Cell 5] Wrote QUBO artifact → data/qubo/scenario_A_qubo.json


,nct_id,phase,score_for_reduction
0,NCT05672537,Phase 2,0.705882
1,NCT05823714,Phase 2,0.705882
2,NCT05823623,Phase 2,0.705882
3,NCT05822752,Phase 2,0.705882
4,NCT05821738,Phase 2,0.705882


### What Cell 4 Just Did

This cell reduced Scenario A down to a tractable number of binary decision variables, then constructed a dense QUBO matrix.

The diagonal encodes the per-trial tradeoff:
- reward benefit (via a negative term),
- penalize cost, and
- penalize safety risk.

An “exactly K selections” constraint is implemented as a quadratic penalty `P·(Σx − K)²`, adding both diagonal and off-diagonal couplings. The full QUBO is saved as `data/qubo/scenario_A_qubo.json` alongside the variable-to-trial mapping.


In [6]:
# ============================================================
# Cell 5 — Solve QUBO with a classical baseline
# ============================================================
#
# What this cell does:
#   1) Defines a QUBO cost function: x^T Q x
#   2) If N is small enough (<=20), runs an exact exhaustive solver.
#   3) Otherwise runs a simple greedy local-improvement procedure.
#
# Why it matters:
#   - Provides a baseline solution that is easy to reproduce and compare.
#   - Lets us validate "the pipeline works" before using quantum resources.
# ============================================================

def qubo_cost(Qmat: np.ndarray, x: np.ndarray) -> float:
    """Compute the QUBO objective value x^T Q x."""
    return float(x @ Qmat @ x)

def exact_solve(Qmat: np.ndarray):
    """
    Exhaustive search over all 2^N bitstrings.
    Only feasible for small N (e.g., N<=20).
    """
    N = Qmat.shape[0]
    best_cost = float("inf")
    best_x = None

    for mask in range(1 << N):
        x = np.fromiter(((mask >> i) & 1 for i in range(N)), count=N, dtype=np.int8)
        c = qubo_cost(Qmat, x)
        if c < best_cost:
            best_cost = c
            best_x = x

    return best_x.astype(int), best_cost

def greedy_local(Qmat: np.ndarray):
    """
    Greedy local-improvement solver:
      - Start from all zeros.
      - Flip the best single bit that reduces cost.
      - Repeat until no single-bit improvement exists.
    """
    N = Qmat.shape[0]
    x = np.zeros(N, dtype=int)
    cur = qubo_cost(Qmat, x)

    improved = True
    while improved:
        improved = False
        best_i = None
        best_new = cur

        for i in range(N):
            x2 = x.copy()
            x2[i] = 1 - x2[i]  # flip bit i
            c = qubo_cost(Qmat, x2)
            if c < best_new:
                best_new = c
                best_i = i

        if best_i is not None:
            x[best_i] = 1 - x[best_i]
            cur = best_new
            improved = True

    return x, cur

log(f"[Cell 6] Solving QUBO classically (N={N})...")

if N <= 20:
    x_best, cost_best = exact_solve(Q)
    method = "exact"
else:
    x_best, cost_best = greedy_local(Q)
    method = "greedy_local"

k_sel = int(x_best.sum())

log(f"[Cell 6] Done | method={method} | QUBO cost={cost_best:.6f} | selected={k_sel} (target K={PORTFOLIO_K})")


15:57:04 | [Cell 6] Solving QUBO classically (N=18)...
15:57:05 | [Cell 6] Done | method=exact | QUBO cost=-1052.117647 | selected=3 (target K=6)


### What Cell 5 Just Did

This cell produced a classical baseline solution to the Scenario A QUBO. If the reduced QUBO is small (≤ 20 variables), it uses an exact exhaustive search. Otherwise, it falls back to a greedy local-improvement method that flips bits until no single flip improves cost.

The result is a best-found bitstring (`x_best`), the associated QUBO objective value, and the number of selected trials implied by that bitstring.


In [7]:
# ============================================================
# Cell 6 — Persist baseline outputs (GitHub-safe)
# ============================================================
#
# What this cell does:
#   1) Converts the chosen bitstring into selected trial IDs.
#   2) Writes:
#        - selected trials CSV (human-readable)
#        - summary CSV (run metadata + objective cost)
#        - bitstring file (exact reproducibility)
#
# Why it matters:
#   - These artifacts are small and can be tracked in GitHub.
#   - They provide a clean baseline for later QAOA comparison.
# ============================================================

# Identify selected indices based on bitstring
selected_idx = np.where(x_best == 1)[0].tolist()
selected_nct = [nct_ids[i] for i in selected_idx]

# Build a selected trials table for inspection
df_selected = df_red.loc[selected_idx].copy()

# Sort by reduction score for convenience (not part of solver)
df_selected = df_selected.sort_values("score_for_reduction", ascending=False)

# Convert bitstring to text (qubit0-first convention here = index order)
bitstring = "".join(str(int(b)) for b in x_best.tolist())

# Output paths
bitstring_path = RESULTS_DIR / "scenario_A_best_bitstring.txt"
selected_csv   = RESULTS_DIR / "scenario_A_classical_selected_trials.csv"
summary_csv    = RESULTS_DIR / "scenario_A_classical_summary.csv"

# Persist bitstring
bitstring_path.write_text(bitstring)

# Pick a compact set of columns (only those that exist)
keep_cols = [c for c in [
    "nct_id",
    "brief_title",
    "phase",
    "lead_sponsor",
    "region_label",
    "score_for_reduction",
    "benefit_work",
    "cost_work",
    "safety_work",
] if c in df_selected.columns]

df_selected[keep_cols].to_csv(selected_csv, index=False)

# Persist a compact summary record
df_summary = pd.DataFrame([{
    "scenario": "A",
    "n_variables": N,
    "portfolio_k_target": PORTFOLIO_K,
    "selected_count": k_sel,
    "solver_method": method,
    "qubo_cost": cost_best,
    "qubo_path": str(qubo_path),
    "selected_csv": str(selected_csv),
    "bitstring_txt": str(bitstring_path),
    "created_utc": datetime.utcnow().isoformat() + "Z",
}])

df_summary.to_csv(summary_csv, index=False)

log(f"[Cell 7] Wrote selected trials → {selected_csv}")
log(f"[Cell 7] Wrote summary         → {summary_csv}")
log(f"[Cell 7] Wrote bitstring       → {bitstring_path}")

df_selected[keep_cols].head(10)


15:57:09 | [Cell 7] Wrote selected trials → data/results/scenario_A_classical_selected_trials.csv
15:57:09 | [Cell 7] Wrote summary         → data/results/scenario_A_classical_summary.csv
15:57:09 | [Cell 7] Wrote bitstring       → data/results/scenario_A_best_bitstring.txt


,nct_id,brief_title,phase,lead_sponsor,region_label,score_for_reduction,benefit_work,cost_work,safety_work
0,NCT05672537,Durvalumab Combined With GemCis Neoadjuvant Th...,Phase 2,Tianjin Medical University Cancer Institute an...,Global / Multi-Region,0.705882,0.705882,0.0,0.0
1,NCT05823714,Venetoclax+Azacytidine+Modified BUCY Condition...,Phase 2,The First Affiliated Hospital of Soochow Unive...,Global / Multi-Region,0.705882,0.705882,0.0,0.0
2,NCT05823623,Inetetamab Combined With Pyrotinib Plus Oral V...,Phase 2,The First Affiliated Hospital with Nanjing Med...,Global / Multi-Region,0.705882,0.705882,0.0,0.0


### What Cell 6 Just Did

This cell translated the optimized bitstring into outputs that are easy to inspect and reproduce. It saved:
- the selected trials as a CSV,
- a one-row summary describing the run configuration and QUBO cost, and
- the raw bitstring used to generate the selection.

These artifacts are intentionally small so they are safe to commit, while the large intermediate tables remain excluded via `.gitignore`.


## Notebook Summary (03e)

In this notebook, we:
1. Loaded Scenario A’s trial universe and the scenario-ready feature table.
2. Standardized benefit, cost, and safety risk to a common scale.
3. Reduced Scenario A to a tractable set of binary variables.
4. Constructed a dense QUBO with an “exactly K” selection constraint.
5. Solved the QUBO using a classical baseline method.
6. Persisted small, reproducible artifacts for comparison and future quantum runs.

### Artifacts produced
- `data/qubo/scenario_A_qubo.json`
- `data/results/scenario_A_classical_selected_trials.csv`
- `data/results/scenario_A_classical_summary.csv`
- `data/results/scenario_A_best_bitstring.txt`

### Next steps
- Review selected trials for face validity (are they sensible given objective tradeoffs?).
- Tune `LAMBDA_COST`, `LAMBDA_SAFETY`, and `PENALTY_K` to reflect realistic priorities.
- Mirror this QUBO into a QAOA workflow (local simulator first, then SV1) and compare against the classical baseline.
